# Assignment 
- Question: What are the top 10 songs played in the top 50 longest sessions by tracks 
count?  
- In this assignment, a user "session" consists of one or more songs played by a given 
user, where each song is started within 20 minutes of the previous song's start time. 

In [1]:
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, StructField, StructType
from pyspark.sql.window import Window

In [2]:
spark = SparkSession.builder.getOrCreate()

In [3]:
DATA_FILE = "../data/lastfm-dataset-1K/userid-timestamp-artid-artname-traid-traname.tsv"
OUTPUT_FILE = "../data/results/excercise2_top_10_songs_in_top_50_longest_sessions.tsv"

SCHEMA = StructType([
    StructField("user_id", StringType()),
    StructField("timestamp_str", StringType()),
    StructField("artist_id", StringType()),
    StructField("artist_name", StringType()),
    StructField("track_id", StringType()),
    StructField("track_name", StringType()),
])

SESSION_GAP_IN_MIN = 20

In [4]:
df = spark.read.csv(DATA_FILE, sep="\t", header=False, schema=SCHEMA)

In [5]:
df.show()

+-----------+--------------------+--------------------+---------------+--------------------+--------------------+
|    user_id|       timestamp_str|           artist_id|    artist_name|            track_id|          track_name|
+-----------+--------------------+--------------------+---------------+--------------------+--------------------+
|user_000001|2009-05-04T23:08:57Z|f1b1cf71-bd35-4e9...|      Deep Dish|                NULL|Fuck Me Im Famous...|
|user_000001|2009-05-04T13:54:10Z|a7f7df4a-77d8-4f1...|       坂本龍一|                NULL|Composition 0919 ...|
|user_000001|2009-05-04T13:52:04Z|a7f7df4a-77d8-4f1...|       坂本龍一|                NULL|Mc2 (Live_2009_4_15)|
|user_000001|2009-05-04T13:42:52Z|a7f7df4a-77d8-4f1...|       坂本龍一|                NULL|Hibari (Live_2009...|
|user_000001|2009-05-04T13:42:11Z|a7f7df4a-77d8-4f1...|       坂本龍一|                NULL|Mc1 (Live_2009_4_15)|
|user_000001|2009-05-04T13:38:31Z|a7f7df4a-77d8-4f1...|       坂本龍一|                NULL|To Stanford (Liv

In [6]:
df = df \
    .withColumn("started_at", F.to_timestamp("timestamp_str")) \
    .withColumn("session", F.session_window("started_at", F.lit(f"{SESSION_GAP_IN_MIN} minutes")))

In [7]:
df.show()

+-----------+--------------------+--------------------+---------------+--------------------+--------------------+-------------------+--------------------+
|    user_id|       timestamp_str|           artist_id|    artist_name|            track_id|          track_name|         started_at|             session|
+-----------+--------------------+--------------------+---------------+--------------------+--------------------+-------------------+--------------------+
|user_000001|2009-05-04T23:08:57Z|f1b1cf71-bd35-4e9...|      Deep Dish|                NULL|Fuck Me Im Famous...|2009-05-04 23:08:57|{2009-05-04 23:08...|
|user_000001|2009-05-04T13:54:10Z|a7f7df4a-77d8-4f1...|       坂本龍一|                NULL|Composition 0919 ...|2009-05-04 13:54:10|{2009-05-04 13:54...|
|user_000001|2009-05-04T13:52:04Z|a7f7df4a-77d8-4f1...|       坂本龍一|                NULL|Mc2 (Live_2009_4_15)|2009-05-04 13:52:04|{2009-05-04 13:52...|
|user_000001|2009-05-04T13:42:52Z|a7f7df4a-77d8-4f1...|       坂本龍一|           

In [8]:
top_50_longest_sessions = df.groupBy("user_id", "session") \
    .agg(
        F.min("started_at").alias("session_start"),
        F.max("started_at").alias("session_end"),
        (F.unix_timestamp(F.max("started_at")) - F.unix_timestamp(F.min("started_at"))).alias("session_duration") 
    ) \
    .orderBy("session_duration", ascending=False) \
    .limit(50)

In [9]:
top_50_longest_sessions.show(50, truncate=False)

+-----------+------------------------------------------+-------------------+-------------------+----------------+
|user_id    |session                                   |session_start      |session_end        |session_duration|
+-----------+------------------------------------------+-------------------+-------------------+----------------+
|user_000949|{2006-02-12 17:49:31, 2006-02-27 11:49:37}|2006-02-12 17:49:31|2006-02-27 11:29:37|1273206         |
|user_000997|{2007-04-26 00:36:02, 2007-05-10 18:15:03}|2007-04-26 00:36:02|2007-05-10 17:55:03|1271941         |
|user_000949|{2007-05-01 02:41:15, 2007-05-14 00:25:52}|2007-05-01 02:41:15|2007-05-14 00:05:52|1113877         |
|user_000544|{2007-02-12 13:03:52, 2007-02-23 01:11:08}|2007-02-12 13:03:52|2007-02-23 00:51:08|906436          |
|user_000949|{2005-12-09 08:26:38, 2005-12-18 05:00:04}|2005-12-09 08:26:38|2005-12-18 04:40:04|764006          |
|user_000949|{2005-11-11 03:30:37, 2005-11-18 23:10:07}|2005-11-11 03:30:37|2005-11-18 2

In [10]:
# group by track_name because track_id can be empty
top_10_songs_in_top_50_longest_sessions = df.alias("base") \
    .join(
        top_50_longest_sessions.alias("top"), 
        on=[
            F.col("base.user_id") == F.col("top.user_id"),
            F.col("base.started_at") >= F.col("top.session_start"),
            F.col("base.started_at") <= F.col("top.session_end")], 
            how="inner"
    ) \
    .groupBy("track_name", "artist_name") \
    .agg(F.count("*").alias("times_played")) \
    .orderBy("times_played", ascending=False) \
    .limit(10)

In [11]:
top_10_songs_in_top_50_longest_sessions.show(truncate=False)

+-------------------------------------+-------------------------+------------+
|track_name                           |artist_name              |times_played|
+-------------------------------------+-------------------------+------------+
|Jolene                               |Cake                     |1215        |
|Heartbeats                           |The Knife                |864         |
|How Long Will It Take                |Jeff Buckley & Gary Lucas|809         |
|Anthems For A Seventeen Year Old Girl|Broken Social Scene      |659         |
|St. Ides Heaven                      |Elliott Smith            |646         |
|Bonus Track                          |The Killers              |634         |
|Starin' Through My Rear View         |2Pac                     |616         |
|Beast Of Burden                      |The Rolling Stones       |613         |
|The Swing                            |Everclear                |604         |
|When You Were Young                  |The Killers  

In [12]:
top_10_songs_in_top_50_longest_sessions.write.mode("overwrite").csv(OUTPUT_FILE, sep="\t", header=True)